In [1]:
import warnings
from IPython.display import Markdown, display
warnings.filterwarnings("ignore")

In [2]:
import pdfplumber as pdf

In [3]:
filename = "Muhammad-pages.pdf"
dictionary_for_pages=[]
with pdf.open(f"../data/building/{filename}") as pdfFile:
    for pageNo,content in enumerate(pdfFile.pages):
        dictionary_for_pages.append({"pageNo":pageNo,"text" : content.extract_text()})
        


In [5]:
import re
chunks = []
chunkNumber = 1

heading = "Default Heading"
current_text = []
startPage = None
endPage = None

for singleDictionary in dictionary_for_pages:

    pageNo = singleDictionary["pageNo"]
    pageText = singleDictionary["text"]

    lines = pageText.split("\n")

    for line in lines:
        line = line.strip()

        if not line:
            continue

        cleaned_line = re.sub(r'\[\d+\]', '', line)
        words = cleaned_line.split()

        # heading detected
        if len(words) < 8 and not cleaned_line.endswith("."):

            if current_text:
                chunks.append({
                    "startPage": startPage,
                    "endPage": endPage,
                    "chunkNumber": chunkNumber,
                    "heading": heading,
                    "chunk_text": " ".join(current_text).strip()
                })

                chunkNumber += 1

            heading = line
            current_text = []

            # new chunk starts from current page
            startPage = pageNo

        else:
            # first line of chunk
            if startPage is None:
                startPage = pageNo

            # update every time text is added
            endPage = pageNo

            current_text.append(line)


# push last remaining chunk
if current_text:
    chunks.append({
        "startPage": startPage,
        "endPage": endPage,
        "chunkNumber": chunkNumber,
        "heading": heading,
        "chunk_text": " ".join(current_text).strip()
    })

In [6]:
# chunks

In [7]:
from sentence_transformers import SentenceTransformer as ST
model = ST("all-MiniLM-L6-v2")

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 365.80it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
chunksText=[]
for chunk in chunks:
    chunksText.append(chunk["chunk_text"])
chunksText
embeddings = model.encode(chunksText)

In [9]:
embeddings

array([[-0.01479094,  0.15283845,  0.00446697, ..., -0.03008716,
        -0.0549854 ,  0.00385174],
       [ 0.00460467,  0.15764512, -0.04094992, ..., -0.01765833,
        -0.02792246, -0.03608555],
       [-0.01667172,  0.03446221, -0.02355837, ..., -0.06727172,
        -0.04321225, -0.00227335],
       [-0.00614691,  0.05360625, -0.03890175, ..., -0.05852663,
         0.01440349, -0.04789002],
       [-0.03182285,  0.05032117, -0.0300909 , ..., -0.05687704,
         0.03145624, -0.03772489]], shape=(5, 384), dtype=float32)

In [10]:
import faiss 

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

In [15]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv("../.env")
os.getenv("GROQ_API_KEY")
# how to create client groq thing
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)
print(client)

In [16]:
# query changing
def generate_better_query(user_query):
    user_query+="\n write a short hypothetical answer to this question"
    # send the query to groq LLM
    message = {"role": "user","content":user_query}
    response  = client.chat.completions.create(model="llama-3.3-70b-versatile",messages=[message])
    return response.choices[0].message.content

In [17]:
def ask(query):
    message = (query + "\n Answer the above question,only using the text below ,and \
    say I don't know if not found ,and also if you do find an answer then also tell, \
    which page number and chunk number u used,if startpage and end page are different ,then tell both,otherwise only one ")
    
    for i in indices[0]:
        message +=f"Page Number = {chunks[i]["startPage"]}\n"
        message +=f"Page Number = {chunks[i]["endPage"]}\n"
        message +=f"Chunk Number = {chunks[i]["chunkNumber"]}\n"
        message += chunks[i]["chunk_text"] + "\n"
    messages = [{"role": "user", "content": message}]
    response  = client.chat.completions.create(model="llama-3.3-70b-versatile",messages=messages)
    return response

In [41]:
query_01 = "What did he say when asked for protection?"
better_query_01 = generate_better_query(query_01)
print("better query for 1 = ",better_query_01)
query_embeddings_01 =  model.encode([better_query_01])
distances,indices = index.search(query_embeddings_01,3)
response = ask(query_01)

# irrelevant question,to test what it responds
query_02 = "What did he say when refusing to help?"
better_query_02 = generate_better_query(query_02)
print("better query for 2 = ",better_query_02)
query_embeddings_02 =  model.encode([better_query_02])
distances,indices = index.search(query_embeddings_02,3)
response1 = ask(query_02)

query_03 = "What did he say when asked for protection?"
# better_query_03 = generate_better_query(query_03)
# print("better query for 1 = ",better_query_01)
query_embeddings_03 =  model.encode([query_03])
distances,indices = index.search(query_embeddings_03,3)
response2 = ask(query_03)

query_04 = "What did he say when refusing to help?"
# better_query_03 = generate_better_query(query_03)
# print("better query for 1 = ",better_query_01)
query_embeddings_04 =  model.encode([query_04])
distances,indices = index.search(query_embeddings_04,3)
response3 = ask(query_04)

better query for 1 =  When asked for protection, he hesitated for a moment before responding, "I'll do everything in my power to keep you safe, but I need you to trust me and stay close. We'll face whatever is coming our way together, okay?" His voice was firm and reassuring, and he looked at her with a determined gaze, as if to convey that he would stop at nothing to shield her from harm.
better query for 2 =  "I'm sorry, I've got my own priorities to attend to and I just can't take on anything else right now," he said, shaking his head and turning to walk away, leaving no room for further discussion.


In [42]:
responseOfQuery = response.choices[0].message.content
responseOfQuery1 = response1.choices[0].message.content
responseOfQuery2 = response2.choices[0].message.content
responseOfQuery3 = response3.choices[0].message.content



In [43]:
display(Markdown(responseOfQuery))

When asked for protection, Muhammad said nothing is mentioned in the provided text about his exact words. However, it is mentioned that he asked for protection from several people, including Akhnas ibn Shariq, Suhayl ibn Amir, and Mut'im ibn 'Adiy.

The relevant information can be found on Page Number = 1, Chunk Number = 5. 

Since the exact words are not mentioned, a more accurate response would be: 
I don't know. (The page and chunk numbers are provided because the question about protection is discussed in that section, but the exact quote is not available.)

In [44]:
display(Markdown(responseOfQuery1))

I don't know. The provided text does not contain information about someone refusing to help. Page Number = 0, Chunk Number = 1, and other chunks do not provide the required information.

In [45]:
display(Markdown(responseOfQuery2))

When asked for protection, Muhammad said nothing is mentioned in the given text, but when Muhammad asked Mut'im ibn 'Adiy for protection, Mut'im agreed. 
Page Number = 1 
Chunk Number = 5

In [46]:
display(Markdown(responseOfQuery3))

When refusing to help, Akhnas declined, saying that he was only a confederate of the house of Quraysh. 
Page Number = 1 
Chunk Number = 5